# Data Cleaning & Structural Validation

**Dataset:** `Raw_clean_dataset.csv`

**Objective:** Clean and validate the provided customer churn dataset by checking missing values, duplicate records, data types, inconsistent values, invalid values, and possible outliers.

**Submission outputs:**
- Cleaned dataset: `cleaned_customer_churn.csv`
- This Jupyter Notebook


## Step 1 — Import Libraries

Pandas is used for data manipulation and NumPy is used for numerical operations.

In [ ]:
import pandas as pd
import numpy as np


## Step 2 — Load the Actual Raw Dataset

The provided file is `Raw_clean_dataset.csv`. The raw dataset is loaded without changing the original CSV file.

In [ ]:
input_file = "Raw_clean_dataset.csv"

df = pd.read_csv(input_file)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()


## Step 3 — Inspect the Dataset

The actual raw dataset contains **600 rows and 13 columns**.

The columns are:
`CustomerID, Age, Region, TenureMonths, SubscriptionType, MonthlyChargesINR, TotalChargesINR, ContractType, SupportTickets, LatePayments, PaymentMethod, AutoPay, Churn`.


In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()


In [ ]:
print("Statistical summary:")
display(df.describe(include="all"))


## Step 4 — Check Missing Values

Missing values are counted for every column. This helps identify fields that need cleaning or imputation.

In [ ]:
missing_before = df.isnull().sum()

print("Missing values by column:")
print(missing_before)

print("\nTotal missing values:", missing_before.sum())


### Observation from the raw dataset

The raw dataset contains **12 missing values in `Region`**. Other columns do not contain standard CSV null values.

These missing `Region` values will be replaced with the category **`Unknown`** so that customer records are not unnecessarily deleted.


## Step 5 — Check Duplicate Records

Duplicate rows are identified before removing them.

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate records:", duplicate_count)

if duplicate_count > 0:
    display(df[df.duplicated(keep=False)])
else:
    print("No duplicate records found.")


### Observation

The raw dataset contains **0 duplicate rows**, so no duplicate records need to be removed at this stage.


## Step 6 — Standardize Column Headers

Column headers are converted to lowercase and spaces/special separators are standardized to underscores.

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("Standardized column names:")
print(df.columns.tolist())


## Step 7 — Clean Text and Categorical Values

Leading and trailing spaces are removed from text fields. Categorical values are standardized for consistency.

In [ ]:
text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = df[column].str.strip()

categorical_columns = [
    "region",
    "subscriptiontype",
    "contracttype",
    "paymentmethod",
    "autopay",
    "churn"
]

for column in categorical_columns:
    if column in df.columns:
        df[column] = df[column].str.title()

print("Text and categorical values standardized.")


## Step 8 — Handle Missing and `unknown` Values

`Region` has missing values, so they are replaced with `Unknown`.

`TotalChargesINR` contains the text value `unknown`. It is treated as a missing numerical value, converted to `NaN`, and then filled using the median of the valid `TotalChargesINR` values. This preserves the customer records while making the column numeric.

In [ ]:
# Standardize missing Region values
if "region" in df.columns:
    df["region"] = df["region"].fillna("Unknown")

# Convert the 'unknown' text in total charges to a missing value
if "totalchargesinr" in df.columns:
    df["totalchargesinr"] = (
        df["totalchargesinr"]
        .replace(r"(?i)^unknown$", np.nan, regex=True)
    )

print("Missing Region values after treatment:",
      df["region"].isna().sum() if "region" in df.columns else "N/A")

print("Missing Total Charges after converting 'unknown':",
      df["totalchargesinr"].isna().sum() if "totalchargesinr" in df.columns else "N/A")


### Observation

The raw dataset contains **10 `unknown` values in `TotalChargesINR`**.

These are treated as missing numerical values rather than as a valid category.


## Step 9 — Validate and Convert Data Types

Expected numerical columns are converted to numeric data types. Invalid text entries become `NaN`, which can then be identified.

In [ ]:
numeric_columns = [
    "age",
    "tenuremonths",
    "monthlychargesinr",
    "totalchargesinr",
    "supporttickets",
    "latepayments"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

print("Data types after conversion:")
print(df.dtypes)


## Step 10 — Handle Remaining Missing Numerical Values

For numerical columns, any remaining missing values are replaced with the median of that column. Median imputation is used because it is less affected by extreme values than the mean.

In [ ]:
for column in numeric_columns:
    if column in df.columns and df[column].isna().sum() > 0:
        median_value = df[column].median()
        df[column] = df[column].fillna(median_value)
        print(f"{column}: missing values replaced with median = {median_value}")

print("\nMissing values after treatment:")
print(df.isnull().sum())


## Step 11 — Check Invalid Values

The following logical checks are performed:
- Age should be between 0 and 100.
- Tenure should not be negative.
- Charges should not be negative.
- Support tickets and late payments should not be negative.

In [ ]:
invalid_checks = {}

if "age" in df.columns:
    invalid_checks["Invalid age"] = ((df["age"] < 0) | (df["age"] > 100)).sum()

if "tenuremonths" in df.columns:
    invalid_checks["Invalid tenure"] = (df["tenuremonths"] < 0).sum()

if "monthlychargesinr" in df.columns:
    invalid_checks["Invalid monthly charges"] = (df["monthlychargesinr"] < 0).sum()

if "totalchargesinr" in df.columns:
    invalid_checks["Invalid total charges"] = (df["totalchargesinr"] < 0).sum()

if "supporttickets" in df.columns:
    invalid_checks["Invalid support tickets"] = (df["supporttickets"] < 0).sum()

if "latepayments" in df.columns:
    invalid_checks["Invalid late payments"] = (df["latepayments"] < 0).sum()

for check, count in invalid_checks.items():
    print(f"{check}: {int(count)}")


## Step 12 — Detect Possible Outliers Using IQR

The Interquartile Range (IQR) method is used to flag possible outliers.

**Important:** Possible outliers are not automatically deleted. An unusual value can still be a valid customer record, so the values are only flagged for review.

In [ ]:
outlier_columns = [
    "age",
    "tenuremonths",
    "monthlychargesinr",
    "totalchargesinr",
    "supporttickets",
    "latepayments"
]

outlier_results = []

for column in outlier_columns:
    if column in df.columns:
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        mask = (df[column] < lower_bound) | (df[column] > upper_bound)

        outlier_results.append({
            "Column": column,
            "Q1": q1,
            "Q3": q3,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Possible Outliers": int(mask.sum())
        })

outlier_table = pd.DataFrame(outlier_results)
display(outlier_table)


## Step 13 — Remove Duplicate Rows

Duplicate rows are removed as a final structural-cleaning step. The raw dataset has already been checked for duplicates.

In [ ]:
rows_before = len(df)

df = df.drop_duplicates().copy()

rows_after = len(df)

print("Rows before duplicate removal:", rows_before)
print("Rows after duplicate removal:", rows_after)
print("Duplicates removed:", rows_before - rows_after)


## Step 14 — Final Validation

The cleaned dataset is checked again for shape, missing values, duplicates, data types, and a sample of the cleaned records.

In [ ]:
print("FINAL DATASET SHAPE")
print("-------------------")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nFINAL MISSING VALUES")
print("--------------------")
print(df.isnull().sum())

print("\nFINAL DUPLICATE COUNT")
print("---------------------")
print(df.duplicated().sum())

print("\nFINAL DATA TYPES")
print("-----------------")
print(df.dtypes)

print("\nFIRST 5 CLEANED RECORDS")
print("-----------------------")
display(df.head())


## Step 15 — Cleaning Decisions Record

| Issue | Cleaning decision | Reason |
|---|---|---|
| Missing `Region` | Replaced with `Unknown` | Keeps the customer record while clearly indicating unavailable region information |
| `unknown` in `TotalChargesINR` | Converted to missing and filled with median | Makes the field numeric and suitable for analysis |
| Duplicate rows | Checked and removed if present | Prevents repeated records |
| Column headers | Standardized | Consistent naming convention |
| Text values | Stripped and standardized | Improves categorical consistency |
| Numeric fields | Converted to numeric types | Structural validation |
| Invalid numeric values | Checked | Prevents logically impossible values |
| Possible outliers | Detected using IQR and retained | Outliers may be genuine observations, so they are not automatically deleted |


## Step 16 — Export the Cleaned Dataset

The final cleaned data is exported as `cleaned_customer_churn.csv`.

In [ ]:
output_file = "cleaned_customer_churn.csv"

df.to_csv(output_file, index=False)

print(f"Cleaned dataset saved as: {output_file}")


## Step 17 — Final Submission Checklist

The required submission should contain:

1. **`cleaned_customer_churn.csv`** — cleaned dataset
2. **`Data_Cleaning_Structural_Validation.ipynb`** — completed Jupyter Notebook

Supporting files:
- `Raw_clean_dataset.csv` — original raw dataset
- `data_cleaning.py` — Python cleaning script

**Before submission:** run all notebook cells successfully and save the notebook with **Ctrl + S**.
